In [ ]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")
pd.set_option("display.max_columns", None)

df = pd.read_csv("AB_NYC_2019.csv")
print("Shape:", df.shape)
print(df.dtypes)
print(df.isna().sum())

with open("step1_overview.txt", "w") as f:
    f.write(f"Initial shape: {df.shape}\n\n")
    f.write("Data types:\n" + str(df.dtypes) + "\n\n")
    f.write("Missing values per column:\n" + str(df.isna().sum()) + "\n\n")
    f.write("Summary statistics (numeric):\n" + str(df.describe()) + "\n\n")
    f.write(f"Duplicate rows: {df.duplicated().sum()}\n")

missing_before = df.isna().sum()

df["name"] = df["name"].fillna("Unknown")
df["host_name"] = df["host_name"].fillna("Unknown")

df["reviews_per_month"] = df["reviews_per_month"].fillna(0)
df["last_review"] = pd.to_datetime(df["last_review"], errors="coerce")
no_review_flag = df["last_review"].isna()
df["has_reviews"] = ~no_review_flag

missing_after = df.isna().sum()

dupes_removed = df.duplicated().sum()
df = df.drop_duplicates()

before_rows = len(df)

# price == 0 is not a valid Airbnb listing price -> erroneous entry
zero_price_count = (df["price"] == 0).sum()
df = df[df["price"] > 0]

# extreme minimum_nights (e.g. > 365) are data entry errors
extreme_min_nights = (df["minimum_nights"] > 365).sum()
df = df[df["minimum_nights"] <= 365]

# price outliers via IQR method (cap rather than delete to preserve sample size)
Q1 = df["price"].quantile(0.25)
Q3 = df["price"].quantile(0.75)
IQR = Q3 - Q1
upper_bound = Q3 + 3 * IQR  # wide multiplier to keep legitimate luxury listings
price_outliers = (df["price"] > upper_bound).sum()
df["price_capped"] = np.where(df["price"] > upper_bound, upper_bound, df["price"])

after_rows = len(df)

with open("step2_cleaning_log.txt", "w") as f:
    f.write("MISSING VALUES\n")
    f.write(f"Before:\n{missing_before}\n\nAfter handling:\n{missing_after}\n\n")
    f.write(f"Duplicate rows removed: {dupes_removed}\n\n")
    f.write("OUTLIERS / ERRONEOUS ENTRIES\n")
    f.write(f"Rows with price == 0 removed: {zero_price_count}\n")
    f.write(f"Rows with minimum_nights > 365 removed: {extreme_min_nights}\n")
    f.write(f"Upper IQR bound for price: {upper_bound:.2f}\n")
    f.write(f"Listings with price above bound (capped, not removed): {price_outliers}\n")
    f.write(f"Row count before cleaning: {before_rows}, after cleaning: {after_rows}\n")

df["neighbourhood_group"] = df["neighbourhood_group"].astype("category")
df["room_type"] = df["room_type"].astype("category")
